# vLLM submission adapter load smoke

This notebook checks the converted `submission_adapter` against the README evaluation contract: vLLM must be able to load a LoRA adapter containing `adapter_config.json` and `adapter_model.safetensors` for NVIDIA Nemotron-3-Nano-30B.

The environment setup is based on `baseline/cot/phase0_offline_eval/infer_rule_based_adapter_leaderboard_proxy_eval.ipynb`. The adapter source root is fixed to `/kaggle/input/datasets/happymiik/mac-to-peft`, while smoke artifacts are written to `/kaggle/working` because Kaggle input datasets are read-only.


In [ ]:
!pip install -U --no-index --find-links /kaggle/input/datasets/luciankucera/vllm-offline-wheels torch vllm datasets


In [ ]:
import polars as pl
import os
from vllm import LLM, SamplingParams
import torch
from datasets import Dataset
import re
import csv
import json
import math
import hashlib
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

try:
    import kagglehub
except ImportError:
    kagglehub = None
import pandas as pd

torch.__version__


In [ ]:
import os
import re
from pathlib import Path

REPO_ROOT = Path("/kaggle/input/datasets/happymiik/mac-to-peft").resolve()
DEFAULT_ADAPTER_PATH = REPO_ROOT / "mlx_to_peft_exporter/outputs/nemotron_sft_lora_with_cot_v2_mlx_notebook_original_fullrun_v2_peft_export/submission_adapter"
DEFAULT_SMOKE_ROOT = Path("/kaggle/working/vllm_load_smoke")

model_override = os.environ.get("NEMOTRON_VLLM_MODEL_PATH", "").strip()
if model_override:
    MODEL_PATH = model_override
elif kagglehub is not None:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
else:
    raise RuntimeError(
        "Set NEMOTRON_VLLM_MODEL_PATH to a local Nemotron base model path when kagglehub is unavailable."
    )

ADAPTER_PATH = str(DEFAULT_ADAPTER_PATH)
SMOKE_ROOT = DEFAULT_SMOKE_ROOT.resolve()
SMOKE_ARTIFACT_ROOT = SMOKE_ROOT / "artifacts"
SMOKE_SUMMARY_JSON = SMOKE_ARTIFACT_ROOT / "vllm_load_smoke_summary.json"

README_MAX_LORA_RANK = 32
README_MAX_TOKENS = 7680
README_TOP_P = 1.0
README_TEMPERATURE = 0.0
README_MAX_NUM_SEQS = 64
README_GPU_MEMORY_UTILIZATION = 0.85
README_MAX_MODEL_LEN = 8192
SMOKE_MAX_TOKENS = int(os.environ.get("NEMOTRON_VLLM_SMOKE_MAX_TOKENS", "256"))

adapter_path = Path(ADAPTER_PATH).resolve()
required_files = [adapter_path / "adapter_config.json", adapter_path / "adapter_model.safetensors"]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Adapter directory must contain adapter_config.json and adapter_model.safetensors. Missing: "
        + ", ".join(missing)
    )
adapter_config = json.loads((adapter_path / "adapter_config.json").read_text(encoding="utf-8"))
adapter_rank = int(adapter_config.get("r", -1))
if adapter_rank < 1 or adapter_rank > README_MAX_LORA_RANK:
    raise ValueError(
        f"Adapter rank {adapter_rank} exceeds README / metric max_lora_rank={README_MAX_LORA_RANK}."
    )

SMOKE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT={REPO_ROOT}")
print(f"MODEL_PATH={MODEL_PATH}")
print(f"ADAPTER_PATH={adapter_path}")
print(f"ADAPTER_RANK={adapter_rank}")
print(f"BASE_MODEL_NAME_OR_PATH={adapter_config.get('base_model_name_or_path')}")
print(f"SMOKE_SUMMARY_JSON={SMOKE_SUMMARY_JSON}")
print(f"SMOKE_MAX_TOKENS={SMOKE_MAX_TOKENS}")


In [ ]:
prompt_rows = [
    {
        "id": "arith-001",
        "prompt": "Solve carefully. What is 17 + 25? Put only the final answer inside \\boxed{}."
    },
    {
        "id": "roman-001",
        "prompt": "Solve carefully. Convert the Roman numeral XIV to an integer. Put only the final answer inside \\boxed{}."
    },
]

prompts = [row["prompt"] for row in prompt_rows]
pd.DataFrame(prompt_rows)


In [ ]:
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=README_MAX_NUM_SEQS,
    gpu_memory_utilization=README_GPU_MEMORY_UTILIZATION,
    dtype='auto',
    max_model_len=README_MAX_MODEL_LEN,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=README_MAX_LORA_RANK,
    enable_prefix_caching=True,
    enable_chunked_prefill=True,
)

sampling_params = SamplingParams(
    temperature=README_TEMPERATURE,
    top_p=README_TOP_P,
    max_tokens=SMOKE_MAX_TOKENS,
)

generate_start = time.perf_counter()
outputs = llm.generate(
    prompts,
    sampling_params=sampling_params,
    lora_request=LoRARequest('submission_adapter', 1, str(adapter_path)),
)
generate_seconds = time.perf_counter() - generate_start

records = []
for row, output in zip(prompt_rows, outputs):
    candidate = output.outputs[0] if output.outputs else None
    raw_output = candidate.text if candidate is not None else ""
    records.append({
        'id': row['id'],
        'prompt': row['prompt'],
        'raw_output': raw_output,
        'finish_reason': getattr(candidate, 'finish_reason', None),
    })

rows_per_second = len(records) / generate_seconds if generate_seconds > 0 else 0.0
print(f"Generated {len(records)} smoke prompts with README-compatible vLLM LoRA settings")
print(f"Generate wall time: {generate_seconds:.2f}s ({rows_per_second:.2f} rows/s)")


In [ ]:
summary_payload = {
    'created_at': datetime.now(timezone.utc).isoformat(),
    'model_path': str(MODEL_PATH),
    'adapter_path': str(adapter_path),
    'adapter_rank': adapter_rank,
    'prompt_count': len(prompt_rows),
    'smoke_max_tokens': SMOKE_MAX_TOKENS,
    'generate_seconds': generate_seconds,
    'records': records,
}
SMOKE_SUMMARY_JSON.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

assert all(record['raw_output'].strip() for record in records), 'One or more smoke generations were empty.'
display(pd.DataFrame(records)[['id', 'finish_reason', 'raw_output']])
print('vLLM LoRA load + generate smoke completed successfully.')
print(f'Saved smoke summary to {SMOKE_SUMMARY_JSON}')
